In [7]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

combined_datasets = pd.read_csv("combined_dataset.csv")

X_combined = combined_datasets.drop("Activity", axis=1)
y_combined = combined_datasets["Activity"]

# Proper split: 80% train, 20% test
X_comb_train, X_comb_test, y_comb_train, y_comb_test = train_test_split(
    X_combined, y_combined, test_size=0.2, random_state=42, stratify=y_combined
)

rf_combined = RandomForestClassifier(n_estimators=100, random_state=42)
rf_combined.fit(X_comb_train, y_comb_train)
y_comb_pred = rf_combined.predict(X_comb_test)

print("=== Combined Dataset with Proper Split ===")
print(classification_report(y_comb_test, y_comb_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_comb_test, y_comb_pred))
print(f"\nTrain size: {len(X_comb_train)}, Test size: {len(X_comb_test)}")

=== Combined Dataset with Proper Split ===
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       281
           SITTING       0.94      0.96      0.95       263
          STANDING       0.97      0.96      0.96       359
           WALKING       0.97      0.99      0.98       272
WALKING_DOWNSTAIRS       0.99      0.97      0.98       204
  WALKING_UPSTAIRS       0.99      1.00      0.99       215

          accuracy                           0.98      1594
         macro avg       0.98      0.98      0.98      1594
      weighted avg       0.98      0.98      0.98      1594


Confusion Matrix:
[[281   0   0   0   0   0]
 [  0 253   9   1   0   0]
 [  0  15 343   1   0   0]
 [  0   0   3 268   0   1]
 [  0   0   0   5 198   1]
 [  0   0   0   0   1 214]]

Train size: 6375, Test size: 1594


In [8]:
# Train with XGBoost Classifier

import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

combined_datasets = pd.read_csv("combined_dataset.csv")
X_combined = combined_datasets.drop("Activity", axis=1)
y_combined = combined_datasets["Activity"]

X_comb_train, X_comb_test, y_comb_train, y_comb_test = train_test_split(
    X_combined, y_combined, test_size=0.2, random_state=42, stratify=y_combined
)

# Encode labels to numeric values for XGBoost
label_encoder = LabelEncoder()
y_comb_train_encoded = label_encoder.fit_transform(y_comb_train)
y_comb_test_encoded = label_encoder.transform(y_comb_test)

xgb_combined = xgb.XGBClassifier(eval_metric='mlogloss', random_state=42)
xgb_combined.fit(X_comb_train, y_comb_train_encoded)
y_comb_xgb_pred_encoded = xgb_combined.predict(X_comb_test)

# Decode predictions back to original labels
y_comb_xgb_pred = label_encoder.inverse_transform(y_comb_xgb_pred_encoded)

print("=== XGBoost on Combined Dataset with Proper Split ===")
print(classification_report(y_comb_test, y_comb_xgb_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_comb_test, y_comb_xgb_pred))
print(f"\nTrain size: {len(X_comb_train)}, Test size: {len(X_comb_test)}")
print(f"\nLabel mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

=== XGBoost on Combined Dataset with Proper Split ===
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       281
           SITTING       0.97      0.98      0.97       263
          STANDING       0.98      0.97      0.97       359
           WALKING       0.97      0.98      0.97       272
WALKING_DOWNSTAIRS       0.98      0.99      0.98       204
  WALKING_UPSTAIRS       1.00      0.99      0.99       215

          accuracy                           0.98      1594
         macro avg       0.98      0.98      0.98      1594
      weighted avg       0.98      0.98      0.98      1594


Confusion Matrix:
[[281   0   0   0   0   0]
 [  0 257   5   1   0   0]
 [  0   8 348   3   0   0]
 [  0   0   2 266   3   1]
 [  0   0   0   3 201   0]
 [  0   0   0   1   1 213]]

Train size: 6375, Test size: 1594

Label mapping: {'LAYING': np.int64(0), 'SITTING': np.int64(1), 'STANDING': np.int64(2), 'WALKING': np.int64(3), 'WALKING_DOWN

## Model Comparison: Random Forest vs XGBoost

Both models achieve excellent **98% accuracy** on the combined dataset!

### Per-Class Performance Comparison:

| Activity | RF Precision | XGB Precision | RF Recall | XGB Recall |
|----------|--------------|---------------|-----------|------------|
| LAYING | 1.00 | 1.00 | 1.00 | 1.00 |
| SITTING | 0.94 | **0.97** ↑ | 0.96 | **0.98** ↑ |
| STANDING | 0.97 | **0.98** ↑ | 0.96 | **0.97** ↑ |
| WALKING | 0.97 | 0.97 | **0.99** | 0.98 |
| WALKING_DOWNSTAIRS | 0.99 | 0.98 | 0.97 | **0.99** ↑ |
| WALKING_UPSTAIRS | 0.99 | **1.00** ↑ | 1.00 | 0.99 |

### Key Differences:

**XGBoost advantages:**
- Better at SITTING (97% vs 94% precision) - fewer false positives
- Better at STANDING (98% vs 97% precision)
- Better at WALKING_DOWNSTAIRS recall (99% vs 97%)
- Perfect precision on WALKING_UPSTAIRS (100%)

**Random Forest advantages:**
- Slightly better WALKING recall (99% vs 98%)
- Both models perform equally well on LAYING (perfect scores)

### Conclusion:
Both models are production-ready with 98% accuracy. **XGBoost has a slight edge** on SITTING and STANDING classification, which were challenging activities in the original cross-dataset tests. For this dataset, either model would work well, but XGBoost may generalize slightly better.

In [ ]:
#